# Representative Shapes by Embedding Region

For each embedding (UMAP, PHATE), for each modality (cell, nucleus): k-means
clusters the full set of embedded cells (every subclass, every condition,
the same cells `01_umap_by_condition_over_time.ipynb` plots) into a small
number of regions, then picks the one real cell closest to each region's
own centroid and shows its actual shape, cropped from its extracted mask.

This runs once per embedding per modality, not once per notebook - the
embedding space itself doesn't change between notebooks 01-03, only which
subset of it they each plot.

Requires `main.py` to have been run first.


In [ ]:
import os
import sys

import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/01_naive_vs_vem/notebooks/), so the analysis dir - and config.py -
# is one level up, and the repo root - where shared/ lives - is two more
# levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR/MASKS_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)

from shared.load_data import load_full_dataset, export_plot_data
from shared.representative_shapes import export_representative_shapes


## Plot: embedding regions and their representative shapes

For each embedding method, one PDF and one SVG (in its own `svgs/`
subfolder), written to `output/plots/00_representative_shapes/`.


In [ ]:
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '00_representative_shapes')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# how many regions to cluster the embedding into, and how large a crop
# (half-width in pixels) to pull each representative cell's shape from -
# a SEPARATE, smaller value for 'nuc' than for 'cell', since a nucleus
# is much smaller than its whole cell - the same half_width that
# comfortably fits a cell crop would leave a nucleus shape too small to
# make out, surrounded by mostly empty padding. Both are a starting
# guess at "large enough for any {cell,nucleus} in this dataset" -
# crop_cell_shape prints a clear warning if a representative cell's own
# shape doesn't fully fit, so bump the relevant one up if that happens.
# Kept the SAME across every region within one modality's own plot (see
# shared.representative_shapes.export_representative_shapes), so
# regions stay directly size-comparable to each other.
N_REGIONS  = 5
HALF_WIDTH = {'cell': 150, 'nuc': 60}

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality):
    '''
    For each embedding method, load this modality's full embedding (every
    subclass, every condition - no filtering), cluster it into N_REGIONS
    regions, and export each region's representative shape.

    Inputs:
        modality: 'cell' or 'nuc'
    Outputs:
        dfs: {embedding_method: the full merged dataset for this
              modality and embedding method, plus a 'region' column}
              (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} cells")

        df = export_representative_shapes(
            df, config.MASKS_DIR, modality, N_REGIONS, embedding_method.upper(),
            f'{MODALITY_NAMES[modality]} morphology (representative regions)',
            NOTEBOOK_PLOTS_DIR, f'{modality}_{embedding_method}', half_width=HALF_WIDTH[modality],
        )
        dfs[embedding_method] = df

    return dfs


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell')


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc')


## Export plot data

Combines every (modality, embedding method) DataFrame above into one CSV
(`plot_data.csv`, alongside the PDFs/SVGs above) - each cell's embedding
coordinates, region assignment, and metadata (not the representative
shape crops themselves, which don't fit a flat CSV).

In [ ]:
plot_data = {('cell', m): df for m, df in cell_dfs.items()}
plot_data.update({('nuc', m): df for m, df in nuc_dfs.items()})
export_plot_data(plot_data, os.path.join(NOTEBOOK_PLOTS_DIR, 'plot_data.csv'))
